In [1]:

from pathlib import Path
import pandas as pd

BASE = Path(r"D:\RouteRanker\data\raw\pune")
matches = list(BASE.rglob("routes.txt"))

if not matches:
    print("routes.txt not found. Check the extracted dataset folder.")
else:
    GTFS_DIR = matches[0].parent
    data = {}

    for name in ["routes", "stops", "trips", "stop_times"]:
        path = GTFS_DIR / f"{name}.txt"

        if path.exists():
            data[name] = pd.read_csv(path, dtype=str)
            df = data[name]
            print(f"\n{name}.txt: {len(df)} rows, {len(df.columns)} columns")
            print("Missing values:", int(df.isna().sum().sum()))
        else:
            print(f"\n{name}.txt: Not found")



routes.txt: 628 rows, 5 columns
Missing values: 0

stops.txt: 6745 rows, 4 columns
Missing values: 0

trips.txt: 10128 rows, 6 columns
Missing values: 0

stop_times.txt: 429302 rows, 6 columns
Missing values: 0


In [2]:

routes = data["routes"]
stops = data["stops"]
trips = data["trips"]
stop_times = data["stop_times"]

print("Duplicate route IDs:", routes["route_id"].duplicated().sum())
print("Duplicate stop IDs:", stops["stop_id"].duplicated().sum())
print("Duplicate trip IDs:", trips["trip_id"].duplicated().sum())

print("Trips referencing missing routes:",
      (~trips["route_id"].isin(routes["route_id"])).sum())

print("Stop times referencing missing trips:",
      (~stop_times["trip_id"].isin(trips["trip_id"])).sum())

print("Stop times referencing missing stops:",
      (~stop_times["stop_id"].isin(stops["stop_id"])).sum())

print("Trips without stop-time records:",
      (~trips["trip_id"].isin(stop_times["trip_id"])).sum())

print("Duplicate stop-time rows:", stop_times.duplicated().sum())


Duplicate route IDs: 0
Duplicate stop IDs: 0
Duplicate trip IDs: 0
Trips referencing missing routes: 0
Stop times referencing missing trips: 0
Stop times referencing missing stops: 0
Trips without stop-time records: 0
Duplicate stop-time rows: 0


In [3]:

# Pune Dataset Inventory

print("Dataset folder:", GTFS_DIR)

for path in sorted(GTFS_DIR.glob("*.txt")):
    df = pd.read_csv(path, dtype=str)

    print(f"\nFile: {path.name}")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Column names:", list(df.columns))
    print("Missing values:", int(df.isna().sum().sum()))


Dataset folder: D:\RouteRanker\data\raw\pune\bus

File: agency.txt
Rows: 1
Columns: 6
Column names: ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_phone', 'agency_email']
Missing values: 0

File: calendar.txt
Rows: 1
Columns: 10
Column names: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
Missing values: 0

File: feed_info.txt
Rows: 1
Columns: 6
Column names: ['feed_publisher_name', 'feed_publisher_url', 'feed_lang', 'feed_start_date', 'feed_end_date', 'feed_contact_url']
Missing values: 0

File: routes.txt
Rows: 628
Columns: 5
Column names: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type']
Missing values: 0

File: shapes.txt
Rows: 256823
Columns: 4
Column names: ['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence']
Missing values: 0

File: stop_times.txt
Rows: 429302
Columns: 6
Column names: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'st


## Dataset Relationships

This dataset follows the General Transit Feed Specification (GTFS), which organizes public transit information into related files.

- **routes.txt:** Contains route information. Routes are identified by `route_id`.
- **trips.txt:** Contains scheduled trips. The `route_id` connects a trip to a route, while `service_id` identifies its service schedule.
- **stop_times.txt:** Contains scheduled stop times and stop order for each trip. The `trip_id` connects records to trips, and `stop_id` connects them to stops.
- **stops.txt:** Contains stop identifiers, names, and available location information.
- **calendar.txt:** When present, defines regular service days and operating periods.
- **calendar_dates.txt:** When present, specifies additions or exceptions to regular service dates.

### Relationship Validation

The notebook checks whether trips reference valid routes, stop-time records reference valid trips and stops, and trips have stop-time records. Duplicate identifiers and missing values are also examined.
